# Colab Accelerator Validation

Use this notebook in Google Colab to validate the accelerator-facing VOI workflow on CPU, GPU, or TPU runtimes. Before running it, choose **Runtime > Change runtime type** and select the hardware target you want to exercise. The notebook records device visibility, checks a deterministic EVPI calculation against a JAX implementation, and confirms that FPGA and ASIC remain explicit placeholder adapters until real runtimes are added.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get("VOIAGE_REPO_URL", "https://github.com/edithatogo/voiage.git")
REPO_DIR = Path(os.environ.get("VOIAGE_REPO_DIR", "/content/voiage"))

if not (Path.cwd() / "pyproject.toml").exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ci]"])
print(f"Using voiage source at {Path.cwd()}")

In [ ]:
import json
import platform
from datetime import datetime, timezone

import jax
import jax.numpy as jnp
import numpy as np

from voiage.methods.basic import evpi
from voiage.parallel import available_execution_adapters, is_placeholder_execution_adapter

net_benefits = np.array([
    [10.0, 12.0, 11.0],
    [14.0, 13.0, 12.0],
    [9.0, 15.0, 13.0],
    [12.0, 11.0, 16.0],
], dtype=float)

cpu_evpi = evpi(net_benefits)
jax_values = jnp.asarray(net_benefits)
jax_evpi = float(jnp.mean(jnp.max(jax_values, axis=1)) - jnp.max(jnp.mean(jax_values, axis=0)))
np.testing.assert_allclose(cpu_evpi, jax_evpi, rtol=1e-12, atol=1e-12)

devices = jax.devices()
platforms = sorted({device.platform for device in devices})
evidence = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "jax_version": jax.__version__,
    "jax_devices": [str(device) for device in devices],
    "jax_platforms": platforms,
    "has_gpu": "gpu" in platforms,
    "has_tpu": "tpu" in platforms,
    "cpu_evpi": cpu_evpi,
    "jax_evpi": jax_evpi,
    "available_execution_adapters": available_execution_adapters(),
    "fpga_is_placeholder": is_placeholder_execution_adapter("fpga"),
    "asic_is_placeholder": is_placeholder_execution_adapter("asic"),
}

assert evidence["fpga_is_placeholder"] is True
assert evidence["asic_is_placeholder"] is True
Path("colab_accelerator_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
evidence

## Interpreting Results

A CPU-only Colab runtime should report `has_gpu = False` and `has_tpu = False`. A GPU or TPU runtime should expose the corresponding JAX platform after the runtime is selected and restarted. The EVPI parity assertion must pass on every runtime. FPGA and ASIC are intentionally reported as placeholders because the repository exposes stable scheduler names without claiming hardware-backed execution.